# 4 — Robustez, rich-club e o efeito da agregação

Último notebook. Duas perguntas de estrutura e uma de método:

1. **O que a cidade aguenta perder?** (robustez)
2. **As regiões mais ativas falam entre si?** (rich-club)
3. **A homofilia socioeconômica que medimos é real?** (falácia ecológica) — esta é a mais
   importante, e muda a conclusão do projeto.

## 1. Preparação

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 40)

from src.utils import load_config
from src import antenna

CIDADE = "campinas"
config = load_config(CIDADE)
config["spatial"]["download_basemap"] = True   # False para rodar offline

edges_antenna = pd.read_parquet(ROOT / config["data"]["edges_antenna_path"])
antennas = pd.read_parquet(ROOT / config["data"]["antennas_path"])

# net traz nós, fluxos, grafo, dirigido e backbone — tudo construído no notebook 2
net = antenna.build(edges_antenna, antennas, config)
nodes, flows = net.nodes, net.flows

print(f"{net.n_antennas} regiões | {net.G.number_of_edges():,} fluxos | "
      f"{nodes['n_users'].sum():,} moradores")

## 2. Por que a robustez tinha que ser reformulada

A análise clássica de robustez remove nós progressivamente e mede o tamanho da **componente
gigante**: redes com hubs desmoronam sob ataque dirigido e resistem a falhas aleatórias.

Antes de reusar isso, vale testar se faz sentido aqui.

In [ ]:
G = net.G
ordem_forca = [u for u, _ in sorted(G.degree(weight="weight"), key=lambda kv: -kv[1])]

print("removendo as regiões de maior volume:")
for frac in [0.1, 0.3, 0.5, 0.7]:
    k = int(frac * G.number_of_nodes())
    H = G.subgraph([n for n in G.nodes() if n not in set(ordem_forca[:k])])
    gigante = max((len(c) for c in nx.connected_components(H)), default=0)
    print(f"  {frac:.0%} removidos → componente gigante = {gigante}/{H.number_of_nodes()} "
          f"({gigante / max(H.number_of_nodes(), 1):.0%} do que restou)")

**A rede praticamente não fragmenta.** Mesmo removendo 70% das regiões mais importantes, o que
sobra continua conectado (só uma região se solta). Com densidade 0,56 isso era inevitável — sempre
há caminho alternativo.

Ou seja: a métrica clássica mediria sempre a mesma coisa (100%) e não diria nada. Precisamos de uma
medida que capture **degradação**, não ruptura.

### A substituta: eficiência global ponderada

A eficiência global é a média das inversas dos caminhos mínimos entre todos os pares. Usando
**1/peso como custo de travessia**, um corredor de muitas chamadas fica "curto" e um fluxo fraco
fica "longo".

Assim, perder uma região de grande volume não desconecta ninguém — mas obriga a comunicação a
passar por caminhos piores, e a eficiência cai. É exatamente o que queremos medir: **a cidade não
se parte, ela perde capacidade.**

In [ ]:
def curva_eficiencia(G, ordem, max_frac=0.6, passos=20):
    base = antenna.weighted_global_efficiency(G)
    fracs = np.linspace(0, max_frac, passos)
    valores = []
    for f in fracs:
        k = int(f * G.number_of_nodes())
        H = G.subgraph([n for n in G.nodes() if n not in set(ordem[:k])])
        valores.append(antenna.weighted_global_efficiency(H) / base)
    return fracs, np.array(valores)


import random
ordem_aleatoria = list(G.nodes())
random.Random(1).shuffle(ordem_aleatoria)

fracs, ataque = curva_eficiencia(G, ordem_forca)
_, aleatorio = curva_eficiencia(G, ordem_aleatoria)

plt.figure(figsize=(7.5, 5))
plt.plot(fracs, ataque, "o-", color="crimson", label="perde as regiões de maior volume")
plt.plot(fracs, aleatorio, "s-", color="steelblue", label="perde regiões ao acaso")
plt.axhline(0.5, color="gray", ls=":", lw=1)
plt.xlabel("fração das regiões removidas")
plt.ylabel("eficiência de comunicação (relativa à rede intacta)")
plt.title(f"Robustez da rede de regiões — {config['city_name']}")
plt.legend(); plt.show()

colapso = next((f for f, y in zip(fracs, ataque) if y < 0.5), None)
print(f"eficiência cai à metade com {colapso:.0%} das regiões removidas (ataque dirigido)")
print(f"diferença entre as curvas com 20% removidas: "
      f"{np.interp(0.2, fracs, aleatorio) - np.interp(0.2, fracs, ataque):.0%}")

As curvas separam bem: perder **28%** das regiões de maior volume corta a capacidade de comunicação
pela metade, enquanto a perda aleatória nunca chega lá em toda a faixa testada.

Leitura para a cidade: **não existe ponto único de falha** — nenhuma região derruba a rede. Mas a
capacidade é desigualmente distribuída, e as regiões do núcleo (as 42 do s-core, notebook 2)
merecem redundância prioritária em planos de contingência.

## 3. Rich-club: as regiões mais ativas conversam entre si?

O rich-club clássico pergunta se os nós de **grau** alto se conectam preferencialmente entre si.
Como grau não distingue nada aqui, usamos a versão **ponderada** (Opsahl et al., 2008), baseada em
força.

A intuição: pegue as regiões com força acima de um limiar e some o peso trocado entre elas. Compare
com o peso que essas mesmas ligações teriam se levassem as arestas mais pesadas da rede. A razão
$\phi^w$ mede o quanto o "clube" concentra volume.

O modelo nulo **embaralha os pesos sobre a mesma topologia** — apropriado justamente porque a
topologia é quase completa e toda a informação está nos pesos.

In [ ]:
from src.pipeline.advanced import _weighted_rich_club


class Mostrar:
    """Exporter mínimo: mostra a figura em vez de salvar."""
    city_name = config["city_name"]
    def save_figure(self, fig, *a, **k):
        plt.show()


resultado_rc = _weighted_rich_club(G, Mostrar(), config["city_name"])
print(resultado_rc)

**ρ ≈ 2,0** no topo da distribuição de força: as regiões de maior tráfego trocam entre si **o dobro**
do volume esperado se os pesos fossem redistribuídos ao acaso.

Existe um "clube" de regiões centrais que concentra a comunicação — o que reforça a conclusão da
robustez: a rede é resiliente a falhas, mas dependente de um núcleo bem identificado.

## 4. O ponto principal: a homofilia era o que parecia?

A conclusão que o projeto carregava era:

> *"49% das chamadas ligam pessoas do mesmo quintil, contra 26% esperado ao acaso — **1,9×**.
> Existe segregação socioeconômica na comunicação."*

E no notebook 3 medimos, entre regiões, apenas **1,12×**. Alguma coisa não fecha. Vamos investigar
com cuidado, porque a resposta é o achado mais forte do trabalho.

### 4.1 Reproduzindo o cálculo antigo

Primeiro refazemos a conta no nível da pessoa, ponderando por volume de chamadas.

In [ ]:
from src.graph_builder import build_edges_graph, build_user_antenna_map

quintil_da_antena = nodes.set_index("antenna_id")["residence_quintile_state"]
user_antenna = build_user_antenna_map(edges_antenna)
user_quintil = user_antenna.map(quintil_da_antena)

pares = build_edges_graph(edges_antenna)          # pares de pessoas, não-direcionados
pares["q_source"] = pares["source"].map(user_quintil)
pares["q_target"] = pares["target"].map(user_quintil)
pares = pares.dropna(subset=["q_source", "q_target"])

w = pares["q_calls"].to_numpy(dtype=float)
observado = w[(pares["q_source"] == pares["q_target"]).to_numpy()].sum() / w.sum()

print(f"volume entre pessoas do MESMO quintil: {observado:.1%}")
print(f"({len(pares):,} pares de pessoas, incluindo os que moram sob a mesma antena)")

### 4.2 O modelo nulo de antes: embaralhar o quintil entre pessoas

O nulo original sorteava o quintil de cada **pessoa**, mantendo a rede fixa.

In [ ]:
rng = np.random.default_rng(42)
usuarios = pd.Index(user_quintil.index)
codigos_pessoa = user_quintil.to_numpy()
si, ti = usuarios.get_indexer(pares["source"]), usuarios.get_indexer(pares["target"])
ok = (si >= 0) & (ti >= 0)

nulo_pessoa = np.mean([w[ok][perm[si[ok]] == perm[ti[ok]]].sum() / w.sum()
                       for perm in (rng.permutation(codigos_pessoa) for _ in range(50))])

print(f"esperado ao acaso (quintil sorteado entre pessoas): {nulo_pessoa:.1%}")
print(f"razão: {observado / nulo_pessoa:.2f}x")

**2,11×** — reproduzimos o achado antigo (a diferença para o 1,9× original é que aqui ponderamos
por volume de chamadas, e antes se contavam arestas).

### 4.3 O problema desse modelo nulo

Aqui está o ponto. Lembre da observação do notebook 2:

> **o quintil é um atributo da antena, não da pessoa.** Todos os moradores de uma antena têm o
> mesmo quintil, porque ele vem colado à geometria da residência.

Quando o nulo sorteia o quintil **de cada pessoa individualmente**, ele destrói essa estrutura: no
mundo embaralhado, vizinhos de porta passam a ter quintis diferentes. Mas no mundo real isso é
impossível por construção.

E lembre também que **29% dos pares de pessoas moram sob a mesma antena** — e esses pares têm o
mesmo quintil **automaticamente**, sem nenhuma preferência social envolvida.

Ou seja: boa parte do "excesso" de 2,11× não mede afinidade socioeconômica. Mede **o fato de as
pessoas falarem com quem mora perto** — e de quem mora perto ter, necessariamente, a mesma renda.

### 4.4 O modelo nulo correto: embaralhar o quintil entre regiões

A correção é natural: em vez de sortear o quintil de cada pessoa, sorteamos o quintil de cada
**região**, preservando quem mora com quem. Assim, a estrutura territorial permanece intacta e o
que sobrar é preferência socioeconômica de verdade.

In [ ]:
antenas_idx = pd.Index(nodes["antenna_id"])
codigos_regiao = quintil_da_antena.reindex(nodes["antenna_id"]).to_numpy()
ai = antenas_idx.get_indexer(pares["source"].map(user_antenna))
bi = antenas_idx.get_indexer(pares["target"].map(user_antenna))
ok2 = (ai >= 0) & (bi >= 0)

nulo_regiao = np.mean([w[ok2][perm[ai[ok2]] == perm[bi[ok2]]].sum() / w.sum()
                       for perm in (rng.permutation(codigos_regiao) for _ in range(50))])

print(f"esperado ao acaso (quintil sorteado entre REGIÕES): {nulo_regiao:.1%}")
print(f"razão: {observado / nulo_regiao:.2f}x")

In [ ]:
valores = [observado, nulo_pessoa, nulo_regiao]
rotulos = ["observado", "acaso\n(quintil entre pessoas)", "acaso\n(quintil entre regiões)"]

plt.figure(figsize=(8, 5))
barras = plt.bar(rotulos, valores, color=["crimson", "lightgray", "steelblue"])
for barra, valor in zip(barras, valores):
    plt.text(barra.get_x() + barra.get_width() / 2, valor + 0.01, f"{valor:.0%}",
             ha="center", fontsize=12)
plt.ylabel("fração do volume entre pessoas do mesmo quintil")
plt.ylim(0, max(valores) * 1.25)
plt.title(f"Preferência social ou proximidade territorial? — {config['city_name']}")
plt.show()

print(f"razão com o nulo antigo (entre pessoas):  {observado / nulo_pessoa:.2f}x")
print(f"razão com o nulo correto (entre regiões): {observado / nulo_regiao:.2f}x")

### 4.5 A conclusão

**De 2,11× para 1,04×.** Praticamente toda a "segregação socioeconômica na comunicação" desaparece
quando o modelo nulo respeita o fato de que vizinhos compartilham renda.

Traduzindo: **as pessoas não escolhem falar com quem tem a mesma renda. Elas falam com quem está
perto — e quem está perto tem a mesma renda.** A segregação que os dados mostram é
**territorial**, não social.

> **Isso tem nome: falácia ecológica** — concluir sobre indivíduos a partir de dados agregados por
> região. O caso aqui é uma variação: usar um modelo nulo que quebra a estrutura de agregação
> produz um efeito que parece individual, mas é geográfico.

### Por que isso melhora a apresentação

Pode parecer que perdemos o principal resultado. É o contrário — o novo é mais forte e mais
acionável:

| Conclusão antiga | Conclusão nova |
|---|---|
| "As pessoas preferem falar com quem tem renda parecida." | "A desigualdade da comunicação **tem endereço**." |
| Sugere política sobre comportamento individual — difícil de operacionalizar. | Sugere política sobre **território**: transporte, mobilidade, localização de serviços. |
| Não indica onde agir. | Indica exatamente onde: os mapas dos notebooks 3. |

E o que sobrevive à correção é concreto e assimétrico: **q5 se fecha (1,23×), q1 se dispersa
(0,69×)**. As pontas da cidade se comportam de forma oposta — isso não é artefato de modelo nulo,
é estrutura real.

## Síntese do projeto

| | Campinas |
|---|---|
| Regiões / fluxos / densidade | 145 / 5.817 / 0,557 |
| Chamadas que não saem da região | 35,9% |
| Backbone | 656 fluxos (11%) com 62% do volume |
| Macro-regiões funcionais | 5, espacialmente contíguas |
| Gravidade | b = 1,09, R² = 0,27 |
| Auto-preferência q5 / q1 | 1,23 / 0,69 |
| Homofilia individual: nulo ingênuo → territorial | **2,11× → 1,04×** |
| Robustez (eficiência à metade) | 28% das regiões, sob ataque dirigido |
| Rich-club ponderado | ρ ≈ 2,0 |

### Os cinco atos da apresentação

1. **O que estamos olhando** — 145 regiões, 25 mil moradores agregados.
2. **A cidade tem um esqueleto** — o backbone no mapa.
3. **A cidade se divide sozinha** — as macro-regiões contíguas. *Melhor figura.*
4. **A segregação tem endereço** — o gráfico de três barras. *Ato principal.*
5. **O que a cidade aguenta** — robustez e as regiões críticas.